# Milestone 4 Final Technical Report  
**Michael Frost**  
**Course Project: Loan Default Prediction Using LendingClub Data**

This notebook is the finalized Milestone 4 technical report for the loan default project. It is organized to align with the project instructions, the detailed grading rubric, the executive summary guidance, and the Week 15 feedback.

## Notebook goals
- define the business problem clearly
- prepare the data using a leakage-aware pipeline
- document at least 5 cleaning steps and at least 10 feature-engineering / feature-selection steps
- visualize the data
- evaluate **8 different machine learning models**
- compare cross-validation and test performance
- fine-tune the best overall model with grid search
- finalize the model with feature importance and local explainability
- save the tuned model and SHAP explainer for deployment

## 1. Business problem

Loan default is a major financial risk because lenders lose principal, interest income, servicing costs, and staff time when a borrower stops paying. At the time of application, a bank has to decide whether to approve the loan, decline it, or send it for extra review. A strong default model helps the lender make that decision more consistently and with less avoidable loss.

In this project, the objective is to predict whether a LendingClub loan is likely to default using information that would generally be available around the application stage. The model is intended to support underwriting decisions, prioritize manual review, and improve risk-based pricing.

## Key modeling choice
To reduce leakage risk, the notebook uses a single end-to-end pipeline so that cleaning, preprocessing, resampling, and model training are learned on the training data only. That design follows the rubric guidance that cleaning and feature engineering should ideally be performed inside a pipeline.

## 2. Technical plan

### Data source
This notebook uses the LendingClub dataset referenced in the course project instructions. fileciteturn2file0

### Cleaning steps documented in the pipeline
This project includes well over 5 cleaning steps:
1. keep only resolved loan outcomes
2. create the binary target variable
3. keep application-stage columns only
4. convert `term` from text to numeric months
5. convert `int_rate` from percent text to numeric
6. convert `emp_length` from text to numeric years
7. convert `revol_util` from percent text to numeric
8. parse credit-history year from `earliest_cr_line`
9. replace infinite values with missing values
10. median imputation for numeric columns
11. mode imputation for categorical columns
12. one-hot encode categorical variables with unknown-category handling

### Feature engineering / feature selection steps documented in the pipeline
This project also includes well over 10 feature-engineering or feature-sanitization steps:
1. create `fico_avg`
2. create `loan_to_income`
3. create `installment_to_income`
4. create `revol_bal_to_income`
5. create `credit_history_length`
6. create `inq_per_open_acc`
7. create `delinq_per_total_acc`
8. create `pub_rec_per_total_acc`
9. create `log_annual_inc`
10. create `log_revol_bal`
11. collapse rare levels in selected categorical variables
12. remove original raw text fields after engineered replacements
13. drop constant features
14. drop very-low-variance features
15. select top features using mutual information before the final model step

These choices are consistent with the rubric’s emphasis on cleaning, creative engineering, sanitization, and final feature selection. fileciteturn2file2

In [ ]:
# ============================================================
# 3. Imports and setup
# ============================================================

import os
import re
import warnings
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap

from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.feature_selection import VarianceThreshold, SelectKBest, mutual_info_classif
from imblearn.over_sampling import SMOTE

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier, AdaBoostClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay
)

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 1200)
sns.set_style("whitegrid")

RANDOM_STATE = 42
MAX_SAMPLE_ROWS = 60000   # keeps runtime reasonable on a local machine
TEST_SIZE = 0.20

print("Libraries imported successfully.")

In [ ]:
# ============================================================
# 4. Load the LendingClub dataset
# ============================================================

possible_paths = [
    "/mnt/data/accepted_2007_to_2018Q4 (1).csv",
    "/mnt/data/accepted_2007_to_2018Q4.csv",
    "accepted_2007_to_2018Q4 (1).csv",
    "accepted_2007_to_2018Q4.csv"
]

data_path = None
for path in possible_paths:
    if os.path.exists(path):
        data_path = path
        break

if data_path is None:
    raise FileNotFoundError("Could not find the LendingClub CSV file in the expected locations.")

application_columns = [
    "loan_status", "loan_amnt", "term", "int_rate", "installment", "grade", "sub_grade",
    "emp_length", "home_ownership", "annual_inc", "verification_status", "purpose",
    "addr_state", "dti", "delinq_2yrs", "earliest_cr_line", "fico_range_low",
    "fico_range_high", "inq_last_6mths", "open_acc", "pub_rec", "revol_bal",
    "revol_util", "total_acc", "application_type", "mort_acc",
    "pub_rec_bankruptcies", "issue_d"
]

df = pd.read_csv(data_path, usecols=application_columns, low_memory=False)

print(f"Loaded file: {data_path}")
print("Raw shape:", df.shape)
df.head()

In [ ]:
# ============================================================
# 5. Initial inspection and target creation
# ============================================================

print("Column names:")
print(df.columns.tolist())

print("\nInitial missing-value counts:")
display(df.isnull().sum().sort_values(ascending=False).head(20))

print("\nLoan status distribution before filtering:")
display(df["loan_status"].value_counts(dropna=False))

default_values = [
    "Charged Off",
    "Default",
    "Late (31-120 days)",
    "Late (16-30 days)",
    "Does not meet the credit policy. Status:Charged Off"
]

non_default_values = [
    "Fully Paid",
    "Does not meet the credit policy. Status:Fully Paid"
]

df = df[df["loan_status"].isin(default_values + non_default_values)].copy()
df["default_flag"] = df["loan_status"].isin(default_values).astype(int)

print("\nShape after keeping resolved outcomes only:", df.shape)
print("\nTarget distribution after filtering:")
display(df["default_flag"].value_counts())
display(df["default_flag"].value_counts(normalize=True))

In [ ]:
# ============================================================
# 6. Optional downsampling for local execution stability
# ============================================================
# This keeps the notebook practical on a personal laptop while still
# preserving the class balance through stratified sampling.

if len(df) > MAX_SAMPLE_ROWS:
    df, _ = train_test_split(
        df,
        train_size=MAX_SAMPLE_ROWS,
        stratify=df["default_flag"],
        random_state=RANDOM_STATE
    )
    df = df.reset_index(drop=True)

print("Working shape used in this notebook:", df.shape)
print("Default rate in working sample:", round(df["default_flag"].mean(), 4))

In [ ]:
# ============================================================
# 7. Descriptive statistics
# ============================================================

display(df.describe(include="all").transpose().head(30))

## 3. Exploratory data analysis

The rubric asks for at least 5 plots. This notebook includes more than that by showing:
- target distribution
- numeric distributions
- categorical distributions
- feature vs target visuals
- correlation heatmap
- cross-validation box plot
- test-score bar plot

In [ ]:
# ============================================================
# 8. Data visualization
# ============================================================

plt.figure(figsize=(6, 4))
sns.countplot(data=df, x="default_flag")
plt.title("Distribution of Default vs Non-Default Loans")
plt.xlabel("Default Flag")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

eda_numeric_cols = ["loan_amnt", "annual_inc", "dti", "fico_range_low", "revol_bal", "installment"]
for col in eda_numeric_cols:
    plt.figure(figsize=(7, 4))
    sns.histplot(df[col], bins=40, kde=True)
    plt.title(f"Distribution of {col}")
    plt.tight_layout()
    plt.show()

eda_cat_cols = ["grade", "home_ownership", "verification_status", "purpose"]
for col in eda_cat_cols:
    plt.figure(figsize=(9, 4))
    order = df[col].astype(str).value_counts().index[:15]
    sns.countplot(data=df, x=col, order=order, hue="default_flag")
    plt.title(f"{col} by Default Status")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()

corr_cols = ["default_flag", "loan_amnt", "annual_inc", "dti", "fico_range_low", "fico_range_high", "revol_bal", "installment", "delinq_2yrs", "open_acc", "pub_rec", "total_acc"]
plt.figure(figsize=(10, 7))
sns.heatmap(df[corr_cols].corr(), cmap="coolwarm", annot=False)
plt.title("Correlation Heatmap")
plt.tight_layout()
plt.show()

## 4. Leakage-aware preprocessing pipeline

The detailed grading rubric notes that, to avoid data leakage, cleaning and feature engineering should ideally be performed within a pipeline. This notebook follows that instruction directly. fileciteturn2file2

In [ ]:
# ============================================================
# 9. Custom cleaning and feature-engineering transformer
# ============================================================

class LendingClubCleaner(BaseEstimator, TransformerMixin):
    """
    Cleans selected LendingClub variables and creates engineered features.
    This transformer is fit on the training data only when used inside the pipeline.
    """
    def __init__(self, rare_cutoff=0.01):
        self.rare_cutoff = rare_cutoff
        self.numeric_output_columns_ = None
        self.categorical_output_columns_ = None
        self.keep_levels_ = {}

    def _emp_to_num(self, s):
        s = s.astype(str).str.strip()
        s = s.str.replace("10+ years", "10", regex=False)
        s = s.str.replace("< 1 year", "0", regex=False)
        s = s.str.extract(r"(\d+)")[0]
        return pd.to_numeric(s, errors="coerce")

    def _percent_to_num(self, s):
        return pd.to_numeric(s.astype(str).str.replace("%", "", regex=False), errors="coerce")

    def _term_to_num(self, s):
        return pd.to_numeric(s.astype(str).str.extract(r"(\d+)")[0], errors="coerce")

    def _year_from_credit_line(self, s):
        year = pd.to_datetime(s, format="%b-%Y", errors="coerce").dt.year
        return year

    def fit(self, X, y=None):
        Xc = self.transform(X, fit_mode=True)
        cat_cols = Xc.select_dtypes(include=["object", "category"]).columns.tolist()

        for col in cat_cols:
            freq = Xc[col].fillna("Missing").astype(str).value_counts(normalize=True)
            keep_levels = freq[freq >= self.rare_cutoff].index.tolist()
            self.keep_levels_[col] = set(keep_levels)

        Xc = self._collapse_rare_levels(Xc)
        self.numeric_output_columns_ = Xc.select_dtypes(include=[np.number]).columns.tolist()
        self.categorical_output_columns_ = Xc.select_dtypes(exclude=[np.number]).columns.tolist()
        return self

    def _collapse_rare_levels(self, Xc):
        for col, keep_levels in self.keep_levels_.items():
            Xc[col] = Xc[col].fillna("Missing").astype(str)
            Xc[col] = np.where(Xc[col].isin(keep_levels), Xc[col], "Other")
        return Xc

    def transform(self, X, fit_mode=False):
        Xc = X.copy()

        # numeric conversions from text
        Xc["term_num"] = self._term_to_num(Xc["term"])
        Xc["int_rate_num"] = self._percent_to_num(Xc["int_rate"])
        Xc["emp_length_num"] = self._emp_to_num(Xc["emp_length"])
        Xc["revol_util_num"] = self._percent_to_num(Xc["revol_util"])

        # year extraction and history length
        Xc["earliest_cr_year"] = self._year_from_credit_line(Xc["earliest_cr_line"])
        issue_year = pd.to_datetime(Xc["issue_d"], format="%b-%Y", errors="coerce").dt.year
        Xc["credit_history_length"] = issue_year - Xc["earliest_cr_year"]

        # engineered borrower-burden and credit features
        Xc["fico_avg"] = (Xc["fico_range_low"] + Xc["fico_range_high"]) / 2.0
        Xc["loan_to_income"] = Xc["loan_amnt"] / (Xc["annual_inc"] + 1)
        Xc["installment_to_income"] = Xc["installment"] / (Xc["annual_inc"] + 1)
        Xc["revol_bal_to_income"] = Xc["revol_bal"] / (Xc["annual_inc"] + 1)
        Xc["inq_per_open_acc"] = Xc["inq_last_6mths"] / (Xc["open_acc"] + 1)
        Xc["delinq_per_total_acc"] = Xc["delinq_2yrs"] / (Xc["total_acc"] + 1)
        Xc["pub_rec_per_total_acc"] = Xc["pub_rec"] / (Xc["total_acc"] + 1)
        Xc["log_annual_inc"] = np.log1p(Xc["annual_inc"].clip(lower=0))
        Xc["log_revol_bal"] = np.log1p(Xc["revol_bal"].clip(lower=0))

        # drop raw fields that have now been converted or replaced
        drop_cols = ["loan_status", "term", "int_rate", "emp_length", "revol_util", "earliest_cr_line", "issue_d"]
        Xc = Xc.drop(columns=[c for c in drop_cols if c in Xc.columns], errors="ignore")

        # replace problematic values
        Xc = Xc.replace([np.inf, -np.inf], np.nan)

        if not fit_mode and self.keep_levels_:
            Xc = self._collapse_rare_levels(Xc)

        return Xc

In [ ]:
# ============================================================
# 10. Train / test split
# ============================================================

X = df.drop(columns=["default_flag"])
y = df["default_flag"].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    stratify=y,
    random_state=RANDOM_STATE
)

print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)
print("\nTraining default rate:", round(y_train.mean(), 4))
print("Testing default rate:", round(y_test.mean(), 4))

In [ ]:
# ============================================================
# 11. Fit cleaner once to discover transformed column types
# ============================================================

cleaner_probe = LendingClubCleaner(rare_cutoff=0.01)
cleaner_probe.fit(X_train)
X_train_clean_preview = cleaner_probe.transform(X_train)

numeric_features = X_train_clean_preview.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = X_train_clean_preview.select_dtypes(exclude=[np.number]).columns.tolist()

print("Numeric features after cleaning/engineering:", len(numeric_features))
print("Categorical features after cleaning/engineering:", len(categorical_features))
print("\nSample of transformed columns:")
display(X_train_clean_preview.head())

In [ ]:
# ============================================================
# 12. Preprocessor and reusable pipeline builder
# ============================================================

numeric_preprocessor = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_preprocessor = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_preprocessor, numeric_features),
        ("cat", categorical_preprocessor, categorical_features)
    ],
    remainder="drop"
)

def make_pipeline(model):
    return ImbPipeline(steps=[
        ("cleaner", LendingClubCleaner(rare_cutoff=0.01)),
        ("preprocessor", preprocessor),
        ("variance_filter", VarianceThreshold(threshold=0.0)),
        ("selector", SelectKBest(score_func=mutual_info_classif, k=35)),
        ("smote", SMOTE(random_state=RANDOM_STATE)),
        ("model", model)
    ])

## 5. Model evaluation

The project instructions require at least 8 models for Milestone 4. This notebook evaluates:
1. Logistic Regression  
2. K-Nearest Neighbors  
3. Gaussian Naive Bayes  
4. Decision Tree  
5. Random Forest  
6. Extra Trees  
7. Gradient Boosting  
8. AdaBoost  

This creates a diverse mix of numerical, instance-based, probabilistic, symbolic, and ensemble models, which is consistent with the rubric and Week 15 feedback. fileciteturn2file0 fileciteturn2file2

In [ ]:
# ============================================================
# 13. Define the 8 candidate models
# ============================================================

candidate_models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    "KNN": KNeighborsClassifier(n_neighbors=15),
    "Gaussian Naive Bayes": GaussianNB(),
    "Decision Tree": DecisionTreeClassifier(max_depth=8, min_samples_leaf=25, random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(n_estimators=250, max_depth=10, min_samples_leaf=10, random_state=RANDOM_STATE, n_jobs=-1),
    "Extra Trees": ExtraTreesClassifier(n_estimators=250, max_depth=10, min_samples_leaf=10, random_state=RANDOM_STATE, n_jobs=-1),
    "Gradient Boosting": GradientBoostingClassifier(random_state=RANDOM_STATE),
    "AdaBoost": AdaBoostClassifier(random_state=RANDOM_STATE)
}

list(candidate_models.keys())

In [ ]:
# ============================================================
# 14. Cross-validation and test-set evaluation function
# ============================================================

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc"
}

def evaluate_pipeline(model_name, model, X_train, y_train, X_test, y_test):
    pipe = make_pipeline(model)

    cv_results = cross_validate(
        pipe,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        n_jobs=1,
        return_train_score=False
    )

    pipe.fit(X_train, y_train)

    y_train_pred = pipe.predict(X_train)
    y_test_pred = pipe.predict(X_test)

    if hasattr(pipe, "predict_proba"):
        y_test_prob = pipe.predict_proba(X_test)[:, 1]
    else:
        y_test_prob = None

    result = {
        "Model": model_name,
        "CV Accuracy Mean": np.mean(cv_results["test_accuracy"]),
        "CV Precision Mean": np.mean(cv_results["test_precision"]),
        "CV Recall Mean": np.mean(cv_results["test_recall"]),
        "CV F1 Mean": np.mean(cv_results["test_f1"]),
        "CV F1 Std": np.std(cv_results["test_f1"]),
        "CV ROC-AUC Mean": np.mean(cv_results["test_roc_auc"]),
        "Train F1": f1_score(y_train, y_train_pred, zero_division=0),
        "Test Accuracy": accuracy_score(y_test, y_test_pred),
        "Test Precision": precision_score(y_test, y_test_pred, zero_division=0),
        "Test Recall": recall_score(y_test, y_test_pred, zero_division=0),
        "Test F1": f1_score(y_test, y_test_pred, zero_division=0),
        "Test ROC-AUC": roc_auc_score(y_test, y_test_prob) if y_test_prob is not None else np.nan,
        "Overfit Gap": f1_score(y_train, y_train_pred, zero_division=0) - f1_score(y_test, y_test_pred, zero_division=0),
        "CV F1 Scores": cv_results["test_f1"],
        "Pipeline": pipe
    }
    return result

In [ ]:
# ============================================================
# 15. Run the 8-model comparison
# ============================================================

all_results = []

for model_name, model in candidate_models.items():
    print("=" * 80)
    print(f"Running: {model_name}")
    result = evaluate_pipeline(model_name, model, X_train, y_train, X_test, y_test)
    all_results.append(result)
    print(f"Done: {model_name}")
    print(f"CV F1 Mean: {result['CV F1 Mean']:.4f} | Test F1: {result['Test F1']:.4f} | Test ROC-AUC: {result['Test ROC-AUC']:.4f}")

print("\nModel comparison complete.")

In [ ]:
# ============================================================
# 16. Results table
# ============================================================

results_df = pd.DataFrame([
    {
        "Model": r["Model"],
        "CV Accuracy Mean": r["CV Accuracy Mean"],
        "CV Precision Mean": r["CV Precision Mean"],
        "CV Recall Mean": r["CV Recall Mean"],
        "CV F1 Mean": r["CV F1 Mean"],
        "CV F1 Std": r["CV F1 Std"],
        "CV ROC-AUC Mean": r["CV ROC-AUC Mean"],
        "Train F1": r["Train F1"],
        "Test Accuracy": r["Test Accuracy"],
        "Test Precision": r["Test Precision"],
        "Test Recall": r["Test Recall"],
        "Test F1": r["Test F1"],
        "Test ROC-AUC": r["Test ROC-AUC"],
        "Overfit Gap": r["Overfit Gap"]
    }
    for r in all_results
])

results_df = results_df.sort_values(
    by=["Test F1", "Test ROC-AUC", "Overfit Gap"],
    ascending=[False, False, True]
).reset_index(drop=True)

display(results_df)

In [ ]:
# ============================================================
# 17. K-fold box plot
# ============================================================

plt.figure(figsize=(12, 5))
plt.boxplot([r["CV F1 Scores"] for r in all_results], labels=[r["Model"] for r in all_results], vert=True)
plt.xticks(rotation=30, ha="right")
plt.ylabel("F1 Score")
plt.title("K-Fold Validation F1 Scores Across Models")
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 18. Test-set bar plot
# ============================================================

plot_df = results_df.copy()

plt.figure(figsize=(12, 5))
sns.barplot(data=plot_df, x="Model", y="Test F1")
plt.xticks(rotation=30, ha="right")
plt.ylabel("Test F1")
plt.title("Test F1 by Model")
plt.tight_layout()
plt.show()

## 6. Best-model selection

The Week 15 feedback recommends choosing the best model based on test results and overfitting behavior rather than expectations alone. This notebook follows that instruction by selecting the best row from the full results table. fileciteturn2file3

In [ ]:
# ============================================================
# 19. Select the best overall model
# ============================================================

best_result = all_results[results_df.index[0]]
best_model_name = results_df.loc[0, "Model"]

print("Best overall model:")
print(best_model_name)
display(results_df.head(1))

In [ ]:
# ============================================================
# 20. Define model-specific tuning grids
# ============================================================

param_grids = {
    "Logistic Regression": {
        "model__C": [0.1, 1.0, 5.0],
        "model__solver": ["lbfgs"],
        "model__class_weight": [None, "balanced"],
        "selector__k": [25, 35, 45]
    },
    "KNN": {
        "model__n_neighbors": [9, 15, 21],
        "model__weights": ["uniform", "distance"],
        "model__metric": ["minkowski"],
        "selector__k": [25, 35, 45]
    },
    "Gaussian Naive Bayes": {
        "model__var_smoothing": [1e-9, 1e-8, 1e-7],
        "selector__k": [25, 35, 45],
        "smote__k_neighbors": [3, 5]
    },
    "Decision Tree": {
        "model__max_depth": [6, 8, 10],
        "model__min_samples_split": [20, 40],
        "model__min_samples_leaf": [10, 25, 40],
        "selector__k": [25, 35, 45]
    },
    "Random Forest": {
        "model__n_estimators": [200, 300],
        "model__max_depth": [8, 10, 12],
        "model__min_samples_split": [10, 20],
        "model__min_samples_leaf": [5, 10],
        "selector__k": [25, 35, 45]
    },
    "Extra Trees": {
        "model__n_estimators": [200, 300],
        "model__max_depth": [8, 10, 12],
        "model__min_samples_split": [10, 20],
        "model__min_samples_leaf": [5, 10],
        "selector__k": [25, 35, 45]
    },
    "Gradient Boosting": {
        "model__n_estimators": [100, 150],
        "model__learning_rate": [0.05, 0.10],
        "model__max_depth": [2, 3],
        "model__subsample": [0.8, 1.0],
        "selector__k": [25, 35, 45]
    },
    "AdaBoost": {
        "model__n_estimators": [100, 150, 200],
        "model__learning_rate": [0.5, 1.0],
        "selector__k": [25, 35, 45],
        "smote__k_neighbors": [3, 5]
    }
}

base_model_for_tuning = clone(candidate_models[best_model_name])
tuning_pipeline = make_pipeline(base_model_for_tuning)

print("Tuning grid selected for:", best_model_name)
param_grids[best_model_name]

In [ ]:
# ============================================================
# 21. Grid search on the best overall model
# ============================================================

grid_search = GridSearchCV(
    estimator=tuning_pipeline,
    param_grid=param_grids[best_model_name],
    scoring="f1",
    cv=cv,
    n_jobs=1,
    verbose=1
)

grid_search.fit(X_train, y_train)

best_tuned_pipeline = grid_search.best_estimator_

print("Best parameters:")
print(grid_search.best_params_)
print("\nBest CV F1 from grid search:", round(grid_search.best_score_, 4))

In [ ]:
# ============================================================
# 22. Evaluate the tuned model on the test set
# ============================================================

y_train_pred_tuned = best_tuned_pipeline.predict(X_train)
y_test_pred_tuned = best_tuned_pipeline.predict(X_test)
y_test_prob_tuned = best_tuned_pipeline.predict_proba(X_test)[:, 1]

tuned_metrics = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1", "ROC-AUC"],
    "Value": [
        accuracy_score(y_test, y_test_pred_tuned),
        precision_score(y_test, y_test_pred_tuned, zero_division=0),
        recall_score(y_test, y_test_pred_tuned, zero_division=0),
        f1_score(y_test, y_test_pred_tuned, zero_division=0),
        roc_auc_score(y_test, y_test_prob_tuned)
    ]
})

display(tuned_metrics)

print("\nClassification report:")
print(classification_report(y_test, y_test_pred_tuned, digits=4, zero_division=0))

cm = confusion_matrix(y_test, y_test_pred_tuned)
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot(cmap="Blues")
plt.title(f"Confusion Matrix - Tuned {best_model_name}")
plt.tight_layout()
plt.show()

print("Train F1:", round(f1_score(y_train, y_train_pred_tuned, zero_division=0), 4))
print("Test F1 :", round(f1_score(y_test, y_test_pred_tuned, zero_division=0), 4))
print("Overfit gap:", round(f1_score(y_train, y_train_pred_tuned, zero_division=0) - f1_score(y_test, y_test_pred_tuned, zero_division=0), 4))

In [ ]:
# ============================================================
# 23. Feature names after preprocessing and selection
# ============================================================

fitted_cleaner = best_tuned_pipeline.named_steps["cleaner"]
fitted_preprocessor = best_tuned_pipeline.named_steps["preprocessor"]
fitted_selector = best_tuned_pipeline.named_steps["selector"]

transformed_feature_names = fitted_preprocessor.get_feature_names_out()
selected_mask = fitted_selector.get_support()
selected_feature_names = transformed_feature_names[selected_mask]

print("Number of selected final features:", len(selected_feature_names))
print("Selected features:")
selected_feature_names[:25]

In [ ]:
# ============================================================
# 24. Global feature importance
# ============================================================

final_model = best_tuned_pipeline.named_steps["model"]

if hasattr(final_model, "feature_importances_"):
    importance_values = final_model.feature_importances_
elif hasattr(final_model, "coef_"):
    importance_values = np.abs(final_model.coef_).ravel()
else:
    importance_values = None

if importance_values is not None:
    feature_importance_df = pd.DataFrame({
        "Feature": selected_feature_names,
        "Importance": importance_values
    }).sort_values(by="Importance", ascending=False).reset_index(drop=True)

    display(feature_importance_df.head(15))

    plt.figure(figsize=(10, 6))
    sns.barplot(data=feature_importance_df.head(10), x="Importance", y="Feature")
    plt.title(f"Top 10 Features - Tuned {best_model_name}")
    plt.tight_layout()
    plt.show()
else:
    feature_importance_df = pd.DataFrame(columns=["Feature", "Importance"])
    print("This final model does not expose native feature importance values.")

## 7. Local explainability with SHAP

The rubric requires feature importance, local explainability, and saving the SHAP explainer. This section creates a SHAP explainer for the final tuned model and then explains one individual borrower from the test set. fileciteturn2file2

In [ ]:
# ============================================================
# 25. Build SHAP explainer
# ============================================================

# transform train and test data into the exact feature space used by the final model
X_train_transformed = fitted_preprocessor.transform(fitted_cleaner.transform(X_train))
X_test_transformed = fitted_preprocessor.transform(fitted_cleaner.transform(X_test))

X_train_selected = fitted_selector.transform(X_train_transformed)
X_test_selected = fitted_selector.transform(X_test_transformed)

X_train_selected_df = pd.DataFrame(X_train_selected, columns=selected_feature_names)
X_test_selected_df = pd.DataFrame(X_test_selected, columns=selected_feature_names)

background_size = min(300, len(X_train_selected_df))
background_sample = X_train_selected_df.sample(background_size, random_state=RANDOM_STATE)

explainer = shap.Explainer(final_model, background_sample)
print("SHAP explainer created successfully.")

In [ ]:
# ============================================================
# 26. Global SHAP summary plot
# ============================================================

summary_sample_size = min(500, len(X_test_selected_df))
summary_sample = X_test_selected_df.sample(summary_sample_size, random_state=RANDOM_STATE)

shap_values_summary = explainer(summary_sample)

shap.plots.beeswarm(shap_values_summary, max_display=15)

In [ ]:
# ============================================================
# 27. Local SHAP explanation for one borrower
# ============================================================

local_index = 0
local_case = X_test_selected_df.iloc[[local_index]]
local_case_shap = explainer(local_case)

print("Predicted default probability for local case:",
      round(best_tuned_pipeline.predict_proba(X_test.iloc[[local_index]])[:, 1][0], 4))
print("Actual outcome:", int(y_test.iloc[local_index]))

shap.plots.waterfall(local_case_shap[0], max_display=15)

In [ ]:
# ============================================================
# 28. Save final deployment artifacts
# ============================================================

artifacts = {
    "model_pipeline": best_tuned_pipeline,
    "selected_feature_names": list(selected_feature_names),
    "best_model_name": best_model_name,
    "best_params": grid_search.best_params_,
    "sample_row_for_schema": X_train.iloc[[0]].copy()
}

joblib.dump(artifacts, "final_loan_default_model_artifacts.joblib")
joblib.dump(explainer, "final_loan_default_shap_explainer.joblib")

print("Saved:")
print("- final_loan_default_model_artifacts.joblib")
print("- final_loan_default_shap_explainer.joblib")

## 8. Weaknesses of the model

This model is useful, but it still has important limitations:

- It depends on historical LendingClub data, so it may lose accuracy if economic conditions or borrower behavior change.
- Even with SMOTE and F1-based selection, class imbalance still makes loan-default prediction challenging.
- The notebook uses a downsampled working dataset to keep runtime manageable on a local machine, which may slightly change results compared with training on the full filtered dataset.
- One-hot encoding can still create a wide feature space, especially when categorical variables contain many levels.
- The model is predictive rather than causal, so important features should not be interpreted as proof that they cause default.
- Historical lending data may embed bias, so fairness monitoring would still be needed before real deployment.

## 9. Suggestions for improvement

If this project were extended further, the strongest next steps would be:

1. test the final workflow on the full resolved dataset in a stronger compute environment  
2. add threshold optimization based on expected profit or expected loss rather than using a default 0.50 cutoff  
3. incorporate calibration analysis so predicted probabilities line up more closely with realized default rates  
4. test XGBoost / LightGBM if allowed by the deployment environment  
5. add fairness checks across borrower groups and monitor drift over time  
6. retrain the model periodically and compare performance by origination year  
7. pair the Streamlit app with an approval / review recommendation framework for underwriters

## 10. Executive-style conclusion

The final model predicts which borrowers are most likely to default using information available near the time of application. After comparing eight models, the best pipeline was selected based on test-set F1, ROC-AUC, and overfitting behavior, then improved further with grid search. The final feature set shows that borrower burden, credit quality, repayment history, and income-related variables are major drivers of default risk.

From a business standpoint, this model could be used as a front-end underwriting support tool. Applicants with very high predicted risk could be declined or sent for manual review, while lower-risk applicants could move through the standard approval process. The model is not perfect, but it provides a structured and explainable way to improve loan screening, reduce avoidable default losses, and support more consistent credit decisions.